- torch.multiprocessing 是 PyTorch 对 Python 原生 multiprocessing 模块的封装和扩展。它提供了与原生模块 100% 兼容的 API，并针对 PyTorch 张量（Tensor）在进程间的共享和传输进行了深度优化。

以下是 torch.multiprocessing 提供的核心方法、组件及其详细解析：

## 🚀 核心启动与进程管理方法
- mp.spawn(fn, args=(), nprocs=1, join=True, daemon=False)
  - 这是 PyTorch 中最常用的多进程启动工具。它会一次性启动 nprocs 个子进程，并自动将进程索引（rank，从 0 开始）作为第一个参数传递给目标函数 fn。<font color='red'>它完美解决了多进程与 CUDA 的兼容性问题，是单机多卡分布式训练的首选启动方式。</font>
- mp.Process(target=func, args=(...))
  - 与原生 multiprocessing.Process 用法完全一致，<font color='red'>用于创建单个子进程。你可以通过 .start() 启动进程，.join() 等待进程结束。</font>
- mp.set_start_method('spawn')
  - <font color='red'>用于设置多进程的启动方式。</font>在涉及 GPU（CUDA）的 PyTorch 任务中，强烈建议将其设置为 'spawn'（默认值），<font color='red'>因为它会启动一个干净的 Python 解释器进程，避免了 'fork' 模式可能导致的 CUDA 运行时冲突或死锁。</font>

## 🧠 张量（Tensor）共享与通信方法
- tensor.share_memory_()
  - 这是 PyTorch 提供的<font color='red'>将张量移动到共享内存的方法</font>。
    - <font color='green'>调用后，该张量在不同进程间传递时将不再进行内存复制，而是共享同一块物理内存，极大提升了数据交换效率。</font>
- mp.Queue() 和 mp.Pipe()
  -<font color='red'> 提供了进程间通信（IPC）的队列和管道机制</font>。torch.multiprocessing 对它们进行了特殊优化：当你通过 Queue 或 Pipe 发送一个张量时，<font color='red'>PyTorch 会自动将其移动到共享内存（如果尚未移动），并只传递一个句柄给接收进程，避免了昂贵的数据拷贝。</font>

## ⚙️ 共享策略管理方法
这些方法<font color='red'>主要用于管理 CPU 张量</font>的共享内存策略（CUDA 张量有自己独立的共享机制）：
- mp.get_sharing_strategy()：返回当前系统 CPU 张量的共享策略。
- mp.set_sharing_strategy(new_strategy)：设置 CPU 张量的共享策略。常见的策略有：
  - 'file_descriptor'（默认）：<font color='red'>使用文件描述符作为共享内存句柄，效率高但受系统打开文件描述符数量限制。</font>
  - 'file_system'：<font color='red'>使用文件系统来标识共享内存，适合文件描述符受限的环境，但需要防范内存泄漏风险。</font>
- mp.get_all_sharing_strategies()：返回当前系统支持的所有共享策略集合。